# Single-emitter ESM observation series generation

This notebook demonstrates generation and saving of synthetic single-aircraft emitter tracks for the experiments described in `docs/single_emitter_observation_series.md`.

Series entries may now contain multiple in-track radar-mode switches. Per-observation ground-truth labels and the aggregate truth mode sequence are retained for evaluation, but the inference view strips those labels before downstream scoring/classification to avoid leakage.

In [ ]:
from pathlib import Path
import json
import os

from esm_observation_series_generator import generate_observation_series, observations_without_ground_truth


In [ ]:
# Use multiple worker processes so this demo exercises the parallelized series generator.
# Cap the notebook demo to a few workers to avoid oversubscribing shared environments.
worker_count = min(4, os.cpu_count() or 1)

data = generate_observation_series(
    count=25,
    seed=42,
    min_duration_s=1.0,
    max_duration_s=60.0,
    sample_interval_s=0.5,
    mode_switch_probability=0.08,  # sampled between adjacent observations; entries can switch several times
    workers=worker_count,
)

len(data["observation_series"]), data["metadata"]


In [ ]:
data["observation_series"][0]

In [ ]:
first_series = data["observation_series"][0]
inference_rows = observations_without_ground_truth(first_series)
{
    "series_id": first_series["series_id"],
    "observation_count": first_series["observation_count"],
    "duration_s": first_series["duration_s"],
    "mode_shift_sequence_indices": first_series["mode_shift_sequence_indices"],
    "ground_truth_modes": [truth["mode"] for truth in first_series["ground_truth_mode_sequence"][:10]],
    "inference_row_has_ground_truth": "ground_truth_label" in inference_rows[0],
    "first_observation_keys": list(first_series["observations"][0].keys()),
    "first_inference_row_keys": list(inference_rows[0].keys()),
}

In [ ]:
output_path = Path("../generated/demo_esm_observation_series.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
output_path